In [1]:
import maboss
import scanpy as sc
import liana as li
from pathlib import Path
import os

cwd=Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent) 
os.getcwd()

'/home/maxi7524/repositories/project_computational_biology_initial_conditions_pymyboss'

## Loading part


## Remarks 

### loading resources

Here i load RNA data, important is that we need to:
- use proper resource for LIG/REC pairs tokens 
- select proper in aggregate functions (many samples does met criterias )

In [2]:
# Define path to the data prepared by the bash script
cell_fate_dir = "data/maboss/CellFate"

# Check if the directory structure exists
if os.path.exists(os.path.join(cell_fate_dir, "filtered_feature_bc_matrix.h5")):
    print("Starting the loading of Visium spatial data...")

    # sc.read_visium looks by default for a file named 'filtered_feature_bc_matrix.h5'
    # and a 'spatial/' subdirectory inside the provided path
    adata = sc.read_visium(path=cell_fate_dir, count_file="filtered_feature_bc_matrix.h5")

    # Initial cleanup of index names (optional, but recommended for OmniPath/LIANA+ integration)
    adata.var_names_make_unique()

    print("\n[OK] AnnData object has been created successfully!")
    print(adata)
else:
    print(
        f"[ERROR] Files not found in: '{cell_fate_dir}'."
        "\nMake sure the data download script (bash) was executed correctly."
    )

Starting the loading of Visium spatial data...


/tmp/ipykernel_137610/839064779.py:10: FutureWarning: Use `squidpy.read.visium` instead.



[OK] AnnData object has been created successfully!
AnnData object with n_obs × n_vars = 3798 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'


/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


In [3]:
adata

AnnData object with n_obs × n_vars = 3798 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

In [4]:
adata.raw = adata

In [5]:
adata.raw

In [ ]:
# create `raw_counts` layer for copying raw X data
adata.layers['raw_counts'] = adata.X.copy()

Index(['MIR1302-2HG', 'FAM138A', 'OR4F5', 'AL627309.1', 'AL627309.3',
       'AL627309.2', 'AL627309.5', 'AL627309.4', 'AP006222.2', 'AL732372.1',
       ...
       'AC133551.1', 'AC136612.1', 'AC136616.1', 'AC136616.3', 'AC136616.2',
       'AC141272.1', 'AC023491.2', 'AC007325.1', 'AC007325.4', 'AC007325.2'],
      dtype='object', length=36601)

In [ ]:
# Run rank_aggregate
li.mt.rank_aggregate(adata, 
                     groupby='region',
                     resource='consensus',
                     expr_prop=0.1,
                     use_raw = True,           # we change default layer
                    #  layer='raw_counts',            # we apply specific layer
                     verbose=True)

Using provided `resource`.


Using the `counts` layer!
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
Make sure that normalized counts are passed!
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.27 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 3036 samples and 1079 features


/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/functools.py:912: UserWarning: zero-centering a sparse array/matrix densifies it.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:293: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.
/home/maxi7524/micromamba/envs/sc_env/lib/python3.12/site-packages/liana/method/sc/_liana_pipe.py:296: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.


Assuming that counts were `natural` log-normalized!
Running CellPhoneDB


100%|██████████| 1000/1000 [00:02<00:00, 479.33it/s]


Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR


In [5]:
rna.uns.keys()

dict_keys(['lesion_colors', 'log1p', 'region_colors', 'spatial', 'liana_res'])

In [6]:
from MaBossSpatial import SpatialBooleanPipeline 

In [7]:
# Setting up pipeline 

## initialization
pipeline = SpatialBooleanPipeline(
    ## adata file, REMARK: it need to contain liana results
    spatial_data=rna, 
    ## value with liana results, you can find it in adata.uns.keys()
    liana_uns_key='liana_res' 
) 

## loading PyMyBoss model
pipeline.SetMaBossModel(
    mode='pretrained',
    model_name='Creative_name',
    bnd_path='data/maboss/CellFate/CellFateModel.bnd',
    cfg_path='data/maboss/CellFate/CellFateModel.cfg'
)

pipeline.SetSpatialSettings(bandwidth=30.0, cutoff=0.05, kernel="gaussian")

pipeline.SetTimeLags(strategy="experimental", custom_lags={"TNF": 10.0, "FAS": 15.0})

pipeline.SetSimulationSettings(max_time=45.0, delta_t=5.0, sample_count=1000)

Successfully loaded pretrained MaBoSS model 'Creative_name' with 28 nodes.


In [8]:
target_cells = rna.obs_names[:2].tolist()
output_path = "data/maboss/real_simulation_results.csv"

pipeline.RunPipeline(target_cell_ids=target_cells, output_csv_path=output_path)

Computing spatial neighborhood connectivity matrix via LIANA+...
Evaluating biological time lags for active pathways...
                    SPATIAL BOOLEAN PIPELINE CONFIGURATION REPORT

[ COMPATIBILITY ALERT ]
--------------------------------------------------------------------------------
⚠️  Warning: Nomenclature Mismatch Detected!
    -> Intracellular Model Organism : HUMAN (Detected via UPPERCASE nodes)
    -> Spatial Expression Dataset   : MOUSE (Detected via Sentence-Case genes)
    -> Impact: Continuous mapping fields will register 0.0 unless matching case rules are added.

--- 1. MODEL TOPOLOGY & MAPPING STATUS ---
    Total Managed Network Nodes: 28
    Directly Mapped Gene Overlaps: 0
    ⚠️  No precise node-to-gene name mapping detected. Check capitalizations.

--- 2. SPATIAL & COMMUNICATION LAYER ---
    ✅  LIANA+ Connectivity Key   : spatial_connectivities
    ✅  Active Spatial Edges      : 3036 connections resolved in tissue grid.

--- 3. TEMPORAL & TIME-LAG CONFIGURATIO

In [10]:
"""
Diagnostic script to validate data mapping, spatial connectivity, and lag structures
within the loaded MaBossSpatial pipeline.
"""
print("=== KROK 1: Weryfikacja mapowania nazw genów ===")
model_nodes = pipeline.maboss_manager.all_nodes
matched_genes = [node for node in model_nodes if node in pipeline.adata.var_names]
missing_genes = [node for node in model_nodes if node not in pipeline.adata.var_names]

print(f"Wszystkie węzły modelu ({len(model_nodes)}): {model_nodes}")
print(f"Dopasowane geny znalezione w adata ({len(matched_genes)}): {matched_genes}")
print(f"Węzły nieobecne w adata (otrzymają warunek początkowy 0.0): {missing_genes}")

print("\n=== KROK 2: Weryfikacja grafu przestrzennego LIANA+ ===")
conn_matrix = pipeline.adata.obsp[pipeline.spatial_env.connectivity_key]
total_edges = conn_matrix.count_nonzero() if hasattr(conn_matrix, "count_nonzero") else np.count_nonzero(conn_matrix)
print(f"Macierz sąsiedztwa LIANA+ zawiera: {total_edges} aktywnych połączeń przestrzennych między komórkami.")

print("\n=== KROK 3: Weryfikacja konfiguracji lagów czasowych ===")
if pipeline.time_estimator and pipeline.time_estimator.intracellular_lags:
    print("Zarejestrowane wewnątrzkomórkowe opóźnienia dla receptorów:")
    for receptor, lag in pipeline.time_estimator.intracellular_lags.items():
        print(f"  - Receptor: {receptor} -> Lag: {lag} minut")
else:
    print("UWAGA: Brak wyliczonych lagów dla receptorów. Sprawdź, czy nazwy w liana_res pokrywają się z modelem.")

=== KROK 1: Weryfikacja mapowania nazw genów ===
Wszystkie węzły modelu (28): ['ATP', 'Apoptosis', 'BAX', 'BCL2', 'CASP3', 'CASP8', 'Cyt_c', 'DISC_FAS', 'DISC_TNF', 'FADD', 'FASL', 'IKK', 'MOMP', 'MPT', 'NFkB', 'NonACD', 'RIP1', 'RIP1K', 'RIP1ub', 'ROS', 'SMAC', 'Survival', 'TNF', 'TNFR', 'XIAP', 'apoptosome', 'cFLIP', 'cIAP']
Dopasowane geny znalezione w adata (0): []
Węzły nieobecne w adata (otrzymają warunek początkowy 0.0): ['ATP', 'Apoptosis', 'BAX', 'BCL2', 'CASP3', 'CASP8', 'Cyt_c', 'DISC_FAS', 'DISC_TNF', 'FADD', 'FASL', 'IKK', 'MOMP', 'MPT', 'NFkB', 'NonACD', 'RIP1', 'RIP1K', 'RIP1ub', 'ROS', 'SMAC', 'Survival', 'TNF', 'TNFR', 'XIAP', 'apoptosome', 'cFLIP', 'cIAP']

=== KROK 2: Weryfikacja grafu przestrzennego LIANA+ ===
Macierz sąsiedztwa LIANA+ zawiera: 3036 aktywnych połączeń przestrzennych między komórkami.

=== KROK 3: Weryfikacja konfiguracji lagów czasowych ===
Zarejestrowane wewnątrzkomórkowe opóźnienia dla receptorów:
  - Receptor: Lrp1 -> Lag: 10.0 minut
  - Receptor